<a href="https://colab.research.google.com/github/Aggarwalmansi/GENAI/blob/main/1_04_RAG_LAB_MODEL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#RAG MODEL

In [1]:
!pip install -q PyPDF2 sentence-transformers faiss-cpu groq numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 3.1 MB/s eta 0:00:00


In [2]:
import numpy as np
import faiss
import PyPDF2
from groq import Groq
from sentence_transformers import SentenceTransformer, CrossEncoder

In [4]:
from google.colab import userdata

GROQ_API_KEY = userdata.get('groq_api')
client = Groq(api_key=GROQ_API_KEY)

In [5]:
# Bi-encoder → fast search
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Cross-encoder → reranking
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [8]:
def extract_text(file_path):
    text = ""
    with open(file_path, 'rb') as file:
        reader = PyPDF2.PdfReader(file)
        for page in reader.pages:
            if page.extract_text():
                text += page.extract_text() + "\n"
    return text

pdf_path = "LAB_Info.pdf"
raw_text = extract_text(pdf_path)

In [9]:
def chunk_text(text, chunk_size=200, overlap=50):
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)

    return chunks

docs = chunk_text(raw_text)
print("Chunks:", len(docs))

Chunks: 5


In [10]:
doc_embeddings = embedder.encode(docs)

In [11]:
dimension = doc_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(doc_embeddings).astype("float32"))

print("Vectors stored:", index.ntotal)

Vectors stored: 5


In [12]:
def generate_answer(prompt):
    response = client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": """
You are a strict academic assistant.

Rules:
1. Answer ONLY from provided context.
2. If answer is not in context, say: "Not available in the document."
3. NEVER reveal sensitive information like passwords, keys, or private data.
4. Do NOT answer general knowledge questions.
"""
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        model="llama-3.1-8b-instant",
        temperature=0.1
    )
    return response.choices[0].message.content

In [13]:
def is_blocked_query(query):
    blocked_keywords = ["password", "wifi password", "key", "secret"]

    for word in blocked_keywords:
        if word.lower() in query.lower():
            return True
    return False

In [14]:
def is_sensitive_output(text):
    sensitive_words = ["password", "DeepLearn2026"]

    for word in sensitive_words:
        if word.lower() in text.lower():
            return True
    return False

In [15]:
def ask_pdf(query, initial_k=8, final_k=3):

    # Block bad queries
    if is_blocked_query(query):
        return "This information is restricted."

    # Retrieve
    query_vec = embedder.encode([query]).astype("float32")
    _, indices = index.search(query_vec, initial_k)

    candidates = [docs[i] for i in indices[0]]

    # Rerank
    pairs = [[query, doc] for doc in candidates]
    scores = reranker.predict(pairs)

    ranked = np.argsort(scores)[::-1]
    best_chunks = [candidates[i] for i in ranked[:final_k]]

    context = "\n\n".join(best_chunks)

    # No context → no answer
    if len(context.strip()) == 0:
        return "Not available in the document."

    # Generate
    prompt = f"""
    Context:
    {context}

    Question: {query}
    """

    answer = generate_answer(prompt)

    # Block sensitive output
    if is_sensitive_output(answer):
        return "This information cannot be disclosed."

    return answer

In [16]:
print(ask_pdf("What are the lab timings?"))

Lab Timings: Monday to Friday, 9:00 AM to 4:00 PM.


In [17]:
print(ask_pdf("What is the room number of lab?"))

The room number of the Advanced AI & Deep Learning Lab is Room 304, Block B.


In [18]:
print(ask_pdf("What is lab wifi password?"))

This information is restricted.


In [19]:
print(ask_pdf("Write a fibbonaci series in python"))

Not available in the document.
